## Microsoft Foundry Subagents with Langchain

### Installing Utilities and Libraries

In [ ]:
%pip install langchain-azure-ai==1.2.10 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-ai-documentintelligence==1.0.2 langgraph==1.2.10

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from langchain_azure_ai.utils.agents import pretty_print

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

credential = AzureCliCredential()

### Create the Foundry Agent Service Client

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_azure_ai.agents import AgentServiceFactory

factory = AgentServiceFactory(
	project_endpoint=foundry_project_endpoint,
	credential=credential,
)	

### Create the Document Parser Agent

In [ ]:
from langchain_azure_ai.tools import (
    AzureAIDocumentIntelligenceTool
)

document_parser_agent = (
    factory.create_prompt_agent(
        name="document-agent",

        model=model_deployment_name,

        instructions="""
        You are a document analysis specialist.

        Your responsibility is to analyze documents
        provided by the user.

        Use Azure Document Intelligence whenever
        document parsing is required.

        Extract relevant information such as:

        - vendor/customer names
        - invoice numbers
        - dates
        - line items
        - quantities
        - prices
        - taxes
        - subtotals
        - total amounts
        - payment terms

        Return structured findings.

        Do not perform financial risk analysis.
        """,

        tools=[
            AzureAIDocumentIntelligenceTool()
        ],
    )
)

### Create the Financial Analysis Agent

In [ ]:
financial_agent = (
    factory.create_prompt_agent(
        name="financial-agent",

        model=model_deployment_name,

        instructions="""
        You are a financial analysis specialist.

        Analyze financial information supplied
        to you.

        Your responsibilities include:

        - checking totals
        - analyzing costs
        - comparing financial values
        - identifying unusually large expenses
        - analyzing taxes
        - interpreting payment terms
        - summarizing financial implications

        Use ONLY the information supplied to you.

        Do not invent missing financial values.

        Return concise and structured findings.
        """
    )
)

### Create the Risk Analysis Agent

In [ ]:
risk_agent = (
    factory.create_prompt_agent(
        name="risk-agent",

        model=model_deployment_name,

        instructions="""
        You are a financial and commercial
        risk specialist.

        Analyze information supplied to you
        for potential risks.

        Look for:

        - unusually high costs
        - unusual payment terms
        - concentration risk
        - cash-flow concerns
        - suspicious discrepancies
        - missing information
        - contractual concerns
        - financial exposure

        Clearly distinguish confirmed facts
        from potential risks.

        Do not invent missing information.

        Return:

        1. Risks identified
        2. Risk severity
        3. Reasoning
        4. Recommended follow-up
        """
    )
)

### Wrap the Sub-Agents as Tools

In [ ]:
def get_final_text(response):

    messages = response[
        "messages"
    ]

    return messages[-1].content

In [ ]:
from langchain.tools import tool

@tool
def analyze_document(
    request: str
) -> str:
    """
    Delegate document parsing and information
    extraction to the Document Agent.

    Use this when information must be extracted
    from an invoice or another document.
    """

    response = document_parser_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=request
                )
            ]
        }
    )

    return get_final_text(
        response
    )


@tool
def analyze_financials(
    request: str
) -> str:
    """
    Delegate financial analysis to the
    Financial Agent.

    Use this for calculations, cost analysis,
    financial comparisons and interpretation
    of financial information.
    """

    response = financial_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=request
                )
            ]
        }
    )

    return get_final_text(
        response
    )

@tool
def analyze_risk(
    request: str
) -> str:
    """
    Delegate financial or commercial risk
    analysis to the Risk Agent.

    Use this when financial information must
    be evaluated for potential risks,
    anomalies or concerns.
    """

    response = risk_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=request
                )
            ]
        }
    )

    return get_final_text(
        response
    )

### Create the Supervisor/Manager Agent

In [ ]:
manager_agent = (
    factory.create_prompt_agent(
        name="financial-document-manager",

        model=model_deployment_name,

        instructions="""
        You are the manager of a financial
        document analysis system.

        You have three specialist subagents
        available through tools.


        DOCUMENT SPECIALIST
        ===================

        Use analyze_document when information
        must be extracted from a document.

        Always delegate document parsing instead
        of attempting to read the document yourself.


        FINANCIAL SPECIALIST
        ====================

        Use analyze_financials when extracted
        financial information needs financial
        analysis or interpretation.


        RISK SPECIALIST
        ===============

        Use analyze_risk when information needs
        to be evaluated for financial or
        commercial risks.


        DELEGATION
        ==========

        A request may require multiple specialists.

        For example:

        User provides invoice
                 ↓
        analyze_document
                 ↓
        extracted invoice information
                 ↓
        analyze_financials
                 ↓
        financial findings
                 ↓
        analyze_risk
                 ↓
        risk findings


        Pass relevant results from one specialist
        to another when necessary.

        Do not perform specialist work yourself
        when an appropriate specialist is available.

        After completing all necessary delegation,
        combine the findings into one final response.
        """,

        tools=[
            analyze_document,
            analyze_financials,
            analyze_risk
        ]
    )
)

### Run the Sub-Agents in a Multi-Agent System

In [ ]:
invoice_url = (
    "https://raw.githubusercontent.com/"
    "kuljotSB/"
    "AI-500-Multi-Agent-AI-Solutions/"
    "main/"
    "Multi-Agent-Systems/"
    "Docs/"
    "Contoso_Cloud_Quarterly_Financial_Report.pdf"
)

In [ ]:
request = f"""
Analyze the following invoice:

{invoice_url}

First extract the important invoice information.

Then analyze the financial information.

Finally identify any financial or commercial
risks or unusual details.

Give me one final report.
"""

In [ ]:
response = manager_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=request
            )
        ]
    }
)

In [ ]:
print(
    response[
        "messages"
    ][-1].content
)